In [1]:
import pandas as pd
print("Ready!")

Ready!


In [2]:
import pandas as pd

the_df = pd.read_excel("data/THE_2026.xlsx")
qs_df = pd.read_csv("data/QS_2026.csv", encoding='latin1')

print("THE shape:", the_df.shape)
print("QS shape:", qs_df.shape)

THE shape: (2191, 14)
QS shape: (1504, 30)


In [3]:
def clean_rank(val):
    val = str(val).replace('=', '').strip()
    if '-' in val:
        parts = val.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return None
    if '+' in val:
        try:
            return float(val.replace('+', ''))
        except:
            return None
    try:
        return float(val)
    except:
        return None

In [4]:
the_df['world_rank'] = the_df['Rank'].apply(clean_rank)
the_df['Name'] = the_df['Name'].astype(str).str.strip()
the_df['Country'] = the_df['Country'].astype(str).str.strip()
the_df = the_df.drop_duplicates()

print("THE cleaned:", the_df.shape)
print("Missing ranks:", the_df['world_rank'].isnull().sum())

THE cleaned: (2191, 15)
Missing ranks: 0


In [5]:
qs_df['world_rank'] = qs_df['Rank'].apply(clean_rank)
qs_df['Name'] = qs_df['Name'].astype(str).str.strip()
qs_df['Country/Territory'] = qs_df['Country/Territory'].astype(str).str.strip()
qs_df = qs_df.drop_duplicates()

print("QS cleaned:", qs_df.shape)
print("Missing ranks:", qs_df['world_rank'].isnull().sum())

QS cleaned: (1504, 31)
Missing ranks: 0


In [6]:
the_df.to_excel("data/THE_cleaned_2026.xlsx", index=False)
qs_df.to_excel("data/QS_cleaned_2026.xlsx", index=False)
print("✅ Cleaned files saved!")

✅ Cleaned files saved!


In [7]:
def normalize_name(name):
    name = str(name).lower().strip()
    name = name.split('(')[0].strip()
    name = name.replace('the ', '').replace('university of', 'univ of')
    return ' '.join(name.split())

the_df['match_key'] = the_df['Name'].apply(normalize_name)
qs_df['match_key'] = qs_df['Name'].apply(normalize_name)

In [8]:
merged = pd.merge(the_df, qs_df, on='match_key', how='inner', suffixes=('_THE', '_QS'))
merged = merged.drop(columns=['match_key'])

print("Merged shape:", merged.shape)
merged.to_excel("data/merged_QS_THE_2026.xlsx", index=False)
print("✅ Merged file saved!")

Merged shape: (879, 46)
✅ Merged file saved!


In [2]:
import pandas as pd

df = pd.read_excel('data/merged_QS_THE_2026.xlsx')
print(df.columns.tolist())

['Rank_THE', 'Name_THE', 'Country', 'Student Population', 'Students to Staff Ratio', 'International Students', 'Female to Male Ratio', 'Overall Score', 'Teaching', 'Research Environment', 'Research Quality', 'Industry Impact', 'International Outlook', 'Year', 'world_rank_THE', 'Rank_QS', 'Previous Rank', 'Name_QS', 'Country/Territory', 'Region', 'Size', 'Focus', 'Research', 'Status', 'Academic Reputation SCORE', 'Academic Reputation  RANK', 'Employer Reputation SCORE', 'Employer Reputation RANK', 'Faculty Student Ratio SCORE', 'Faculty Student Ratio RANK', 'Citations per Faculty SCORE', 'Citations per Faculty RANK', 'International Faculty  SCORE', 'International Faculty  RANK', 'International Student SCORE', 'International student  RANK', 'International Students Diversity SCORE', 'International Students Diversity RANK', 'International Research Network SCORE', 'International Research Network RANK', 'Employment Outcomes SCORE', 'Employment Outcomes RANK', 'Sustainability SCORE', 'Sustain

In [4]:
cols_to_fix = ['Overall Score', 'Overall SCORE', 'Research Quality', 'Citations per Faculty SCORE',
               'Students to Staff Ratio', 'Faculty Student Ratio SCORE', 'International Students',
               'International Student SCORE', 'Academic Reputation SCORE', 'Employer Reputation SCORE',
               'Research Environment', 'Industry Impact']

for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Global_Ranking_Score'] = df[['Overall Score', 'Overall SCORE']].mean(axis=1)

df['Research_Impact_Score'] = df[['Research Quality', 'Citations per Faculty SCORE']].mean(axis=1)

df['Faculty_to_Student_Ratio'] = df[['Students to Staff Ratio', 'Faculty Student Ratio SCORE']].mean(axis=1)

df['International_Student_Percentage'] = df[['International Students', 'International Student SCORE']].mean(axis=1)

df['Academic_Reputation_Score'] = df[['Academic Reputation SCORE', 'Employer Reputation SCORE']].mean(axis=1)

df['Research_Productivity_Index'] = df[['Research Environment', 'Industry Impact']].mean(axis=1)

df.to_excel('data/university_final_dataset.xlsx', index=False)

print(df[['Global_Ranking_Score', 'Research_Impact_Score', 'Faculty_to_Student_Ratio', 'International_Student_Percentage', 'Academic_Reputation_Score', 'Research_Productivity_Index']].head())

   Global_Ranking_Score  Research_Impact_Score  Faculty_to_Student_Ratio  \
0              98.05500                  94.35                     55.20   
1              98.83675                  99.80                     53.85   
2              93.30550                  99.50                     37.75   
3              97.20325                  92.85                     55.65   
4              98.02550                  99.60                     52.95   

   International_Student_Percentage  Academic_Reputation_Score  \
0                            49.515                      100.0   
1                            45.965                      100.0   
2                            35.115                       99.9   
3                            46.740                      100.0   
4                            36.875                      100.0   

   Research_Productivity_Index  
0                        99.95  
1                        97.65  
2                        97.65  
3             